<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_05_RNNs_and_LSTMs_on_Time_Series_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 5: Time-series Prediction using Recurrent vs. Gated Architectures

**Instructor:** John Sipple

**Frameworks:** JAX, Optax, and Equinox

### Overview

In this tutorial, we will tackle multivariate time-series forecasting. Specifically, our goal is to predict future temperature values based on historical weather features, including atmospheric pressure, air density, and past temperatures.

To provide a comprehensive understanding of both foundational architectures and modern library implementations, we will build these temporal models from the ground up. We will start by implementing a Vanilla Recurrent Neural Network (RNN) and a Long Short-Term Memory (LSTM) network mathematically from scratch using only pure JAX, before transitioning to the high-level, object-oriented library, Equinox.

### The Problem Setup

We frame this forecasting challenge as a Sequence-to-Sequence (Seq2Seq) problem utilizing the open-source Jena Climate dataset.

* **Input Sequence ($X$):** The network observes the last 24 time steps (representing the past 4 hours) across 4 distinct meteorological features.


* **Target Sequence ($Y$):** The network must predict the next 6 time steps (representing the next 1 hour) exclusively for Temperature.



### Key Concepts Explored:

* **Data Processing & Visualization:** Downloading, standardizing, and windowing the Jena Climate dataset. This includes plotting macro-trends and inspecting micro-level sequence windows to visualize exactly what the network processes at a given time step.


* **Vanilla RNN Mechanics:** Implementing a basic RNN from scratch to understand the step-by-step processing of a sequence, the mathematical update rules for the hidden state ($h_t$), and the limitations of a single projection layer.


* **LSTM Encoder-Decoder Architectures:** Upgrading to an LSTM cell to mitigate the vanishing gradient problem by maintaining a secondary Cell State ($C_t$) governed by Forget, Input, and Output gates. You will construct an Encoder to condense the 24 historical steps into a context vector, and a Decoder to recursively generate the 6 future steps.


* **The Transition to Equinox:** Moving from managing manual parameter dictionaries and explicitly tracking matrix shapes in JAX to defining neural networks seamlessly as callable PyTrees using the Equinox `eqx.Module` framework.


* **Evaluation & Prediction:** Comparing the training stability, Mean Squared Error (MSE), and predictive trajectory of the Vanilla RNN versus the gated LSTM architecture.

In [ ]:
# --- Setup and Installation ---
!pip install -q equinox optax jax jaxlib pandas matplotlib scikit-learn

import jax
import jax.numpy as jnp
import equinox as eqx
import optax
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import tensorflow as tf

# Set precision for better gradient stability
jax.config.update("jax_enable_x64", False)

print(f"JAX Devices: {jax.devices()}")
print(f"Equinox Version: {eqx.__version__}")

#Part A: The Jena Climate Dataset

We will use the open-source [Jena Climate dataset](https://www.kaggle.com/datasets/mnassrib/jena-climate) recorded by the Max Planck Institute for Biogeochemistry. It contains 14 features recorded every 10 minutes.

We will frame this as a sequence-to-sequence problem:

* **Input Sequence** ($X$): The last 24 steps (past 4 hours) of 4 selected features.

* **Target Sequence** ($Y$): The next 6 steps (next 1 hour) of Temperature.

In [ ]:
# --- 1. Download and Load Data ---

fname = 'jena_climate_2009_2016.csv.zip'
csv_path = tf.keras.utils.get_file(
    origin='https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname=fname,
    extract=True)
csv_path = os.path.join(csv_path, fname)
csv_path = csv_path.replace('.zip', '')

df = pd.read_csv(csv_path)

# Select a subset of features to keep it lightweight:
# p (mbar), T (degC), rho (g/m**3), VPmax (mbar)
features = ['p (mbar)', 'T (degC)', 'rho (g/m**3)', 'VPmax (mbar)']
data = df[features].values

# We'll use a small subset of the data (e.g., first 20,000 steps) for rapid training in this demo
data = data[:20000]

# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# --- 2. Create Time-Series Windows ---
def create_windows(data, input_steps=24, output_steps=6, target_idx=1):
    X, Y = [], []
    for i in range(len(data) - input_steps - output_steps):
        X.append(data[i : i + input_steps, :])
        # We only want to predict Temperature (index 1)
        Y.append(data[i + input_steps : i + input_steps + output_steps, target_idx])
    return np.array(X), np.array(Y)

INPUT_STEPS = 24
OUTPUT_STEPS = 6
TARGET_IDX = 1 # 'T (degC)'

X, Y = create_windows(data_scaled, INPUT_STEPS, OUTPUT_STEPS, TARGET_IDX)

# Train/Test Split (80/20)
split_idx = int(len(X) * 0.8)
X_train, Y_train = jnp.array(X[:split_idx]), jnp.array(Y[:split_idx])
X_test, Y_test = jnp.array(X[split_idx:]), jnp.array(Y[split_idx:])

print(f"X_train shape: {X_train.shape} (Samples, TimeSteps, Features)")
print(f"Y_train shape: {Y_train.shape} (Samples, FutureSteps)")

##Visualizing the Time-Series Data

Before feeding data into a neural network, it is critical to visualize both the overall trends and the specific structure of the inputs and outputs.


We will look at the data from two perspectives:

1. **Macro View:** Plotting the first 1,000 steps of the unscaled dataset. Notice the inverse correlation between Temperature (T (degC)) and Air Density (rho (g/m**3)), as well as the strong diurnal (daily) cycles.

2. **Micro View (The Sliding Window):** Visualizing a single sample $i$ from our generated dataset.

* The network sees the Past 24 steps ($X_i$) across all 4 features.

* The network must predict the Future 6 steps ($Y_i$) for Temperature only.

* These values are standardized ($\mu = 0, \sigma = 1$), which is how the neural network processes them to avoid gradient instability.

In [ ]:
# --- 1. Macro View: Raw Data Trends ---
def plot_macro_trends(data, features, steps=1000):
    fig, axes = plt.subplots(len(features), 1, figsize=(12, 8), sharex=True)

    for i, feature in enumerate(features):
        axes[i].plot(data[:steps, i], color='tab:blue')
        axes[i].set_ylabel(feature, fontsize=10)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Steps (10 min intervals)", fontsize=12)
    plt.suptitle(f"Macro View: First {steps} Time Steps of Jena Climate Data", fontsize=14)
    plt.tight_layout()
    plt.show()

# Plot using the original (unscaled) data for interpretability
plot_macro_trends(data, features, steps=1000)

# --- 2. Micro View: Sequence-to-Sequence Formulation ---
def plot_training_window(X_data, Y_data, features, target_idx, sample_idx=42):
    # Time axes relative to the prediction point
    past_time = np.arange(-X_data.shape[1], 0)
    future_time = np.arange(0, Y_data.shape[1])

    fig, ax = plt.subplots(figsize=(10, 5))

    # Plot all past features (X)
    for i, feature in enumerate(features):
        if i == target_idx:
            # Highlight the historical target feature
            ax.plot(past_time, X_data[sample_idx, :, i],
                    label=f'Historical {feature}', color='blue', linewidth=2.5)
        else:
            # Other features
            ax.plot(past_time, X_data[sample_idx, :, i],
                    label=f'Past {feature}', linestyle='--', alpha=0.6)

    # Plot future target (Y)
    ax.plot(future_time, Y_data[sample_idx, :],
            label=f'True Future {features[target_idx]}',
            color='red', marker='o', markersize=6, linewidth=2.5)

    # Formatting
    ax.axvline(0, color='black', linestyle='-', linewidth=1.5, alpha=0.8)
    ax.axvspan(0, future_time[-1], color='red', alpha=0.05, label='Prediction Horizon')

    ax.set_title(f"Micro View: Input Window vs. Target Horizon (Standardized Sample {sample_idx})", fontsize=14)
    ax.set_xlabel("Time Steps relative to prediction point", fontsize=12)
    ax.set_ylabel("Standardized Values", fontsize=12)

    # Put legend outside the plot
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot using the scaled data (what the model actually sees)
plot_training_window(X_train, Y_train, features, TARGET_IDX, sample_idx=150)

#Part B: Pure JAX Vanilla Recurrent Neural Network (From Scratch)

Instead of relying on a high-level library, we will implement the Vanilla RNN mathematically from scratch using JAX.

**Mathematical Formulation**

A vanilla RNN processes a sequence step-by-step, maintaining a single hidden state $h_t$.

Given an input vector $x_t \in \mathbb{R}^d$ and the previous hidden state $h_{t-1} \in \mathbb{R}^h$, the update rule for the hidden state is:

$$h_t = \tanh(W_{ih} x_t + W_{hh} h_{t-1} + b_h)$$

After processing the entire sequence of length $T$, the final hidden state $h_T$ summarizes the temporal information. We project this final state through a linear layer to generate our future predictions $\hat{y}$:

$$\hat{y} = W_{ho} h_T + b_o$$

In pure JAX, we manage the state manually by initializing these parameter matrices and passing them explicitly into our forward pass function.

In [ ]:
# --- 1. Pure JAX RNN Initialization ---
def init_vanilla_rnn_params(input_size, hidden_size, output_size, key):
    """Initializes the weight matrices and biases for the Vanilla RNN."""
    k1, k2, k3 = jax.random.split(key, 3)

    # He/Xavier-style initialization scaled by input dimension
    return {
        'W_ih': jax.random.normal(k1, (input_size, hidden_size)) * jnp.sqrt(2.0 / input_size),
        'W_hh': jax.random.normal(k2, (hidden_size, hidden_size)) * jnp.sqrt(2.0 / hidden_size),
        'b_h': jnp.zeros(hidden_size),
        'W_ho': jax.random.normal(k3, (hidden_size, output_size)) * jnp.sqrt(2.0 / hidden_size),
        'b_o': jnp.zeros(output_size)
    }

# --- 2. Pure JAX RNN Forward Pass ---
def rnn_forward(params, x):
    """
    Executes the forward pass of the RNN over a sequence.
    Args:
        params: Dictionary containing the weights and biases.
        x: Input sequence of shape (Sequence_Length, Features).
    """
    hidden_size = params['W_hh'].shape[0]

    # The function applied at each time step
    def scan_fn(h_prev, x_t):
        # h_t = tanh(x_t * W_ih + h_{t-1} * W_hh + b_h)
        h_next = jax.nn.tanh(
            jnp.dot(x_t, params['W_ih']) +
            jnp.dot(h_prev, params['W_hh']) +
            params['b_h']
        )
        # scan expects (carry_over_state, output_for_this_step)
        return h_next, h_next

    # Initialize the hidden state to zeros for the first time step
    h_init = jnp.zeros(hidden_size)

    # jax.lax.scan unrolls the loop highly efficiently in XLA
    h_final, _ = jax.lax.scan(scan_fn, h_init, x)

    # Project the final hidden state to the prediction horizon
    preds = jnp.dot(h_final, params['W_ho']) + params['b_o']
    return preds

# Initialize the parameters
key = jax.random.PRNGKey(42)
rnn_params = init_vanilla_rnn_params(input_size=4, hidden_size=32, output_size=OUTPUT_STEPS, key=key)

## Rendering the Internal RNN Architecture and Weights


To visualize what we just built, the function below inspects the rnn_params dictionary. It dynamically extracts the shapes of the weight matrices and biases to draw the exact tensor transformations occurring inside the RNN cell and the final output projection.

In [ ]:
import matplotlib.patches as patches

def render_pure_rnn(params):
    """
    Renders the internal architecture of the pure JAX RNN,
    annotating the specific shapes of the initialized weights.
    """
    # Extract dimensions
    in_dim, h_dim = params['W_ih'].shape
    out_dim = params['W_ho'].shape[1]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')

    # Main RNN Cell Box
    cell_box = patches.FancyBboxPatch((3, 2), 4, 3, boxstyle="round,pad=0.1",
                                      ec="black", fc="#e6f2ff", lw=2)
    ax.add_patch(cell_box)
    ax.text(5, 5.2, "Vanilla RNN Cell", fontsize=14, weight='bold', ha='center')

    # Input x_t
    ax.arrow(5, 0.5, 0, 0.8, head_width=0.1, fc='k', ec='k', lw=1.5)
    ax.text(5, 0.2, f"$x_t$\nshape: ({in_dim},)", ha='center', va='center', fontsize=12)

    # W_ih multiplication
    ax.add_patch(patches.Rectangle((4.2, 1.5), 1.6, 0.6, color='yellow', ec='black'))
    ax.text(5, 1.8, f"$W_{{ih}}$\n({in_dim}x{h_dim})", ha='center', va='center', fontsize=10)

    # Previous Hidden State h_{t-1}
    ax.arrow(1.5, 3.5, 0.8, 0, head_width=0.1, fc='k', ec='k', lw=1.5)
    ax.text(1.0, 3.5, f"$h_{{t-1}}$\n({h_dim},)", ha='center', va='center', fontsize=12)

    # W_hh multiplication
    ax.add_patch(patches.Rectangle((2.5, 3.2), 1.2, 0.6, color='yellow', ec='black'))
    ax.text(3.1, 3.5, f"$W_{{hh}}$\n({h_dim}x{h_dim})", ha='center', va='center', fontsize=10)

    # Addition Node (incorporating bias)
    ax.add_patch(patches.Circle((5, 3.5), 0.3, fc='lightgreen', ec='black', zorder=3))
    ax.text(5, 3.5, "+", ha='center', va='center', fontsize=14, weight='bold', zorder=4)
    ax.text(5.5, 3.1, f"$+ b_h$ ({h_dim},)", ha='center', va='center', fontsize=10, color='darkgreen')

    # Arrow from W_ih to +
    ax.arrow(5, 2.1, 0, 1.0, head_width=0.1, fc='k', ec='k', lw=1.5)

    # Tanh Activation
    ax.add_patch(patches.Rectangle((4.5, 4.2), 1.0, 0.5, color='pink', ec='black'))
    ax.text(5, 4.45, "tanh", ha='center', va='center', fontsize=11, weight='bold')
    ax.arrow(5, 3.8, 0, 0.3, head_width=0.1, fc='k', ec='k', lw=1.5)

    # Next Hidden State h_t
    ax.arrow(5.5, 4.45, 2.5, 0, head_width=0.1, fc='k', ec='k', lw=1.5)
    ax.text(8.5, 4.45, f"$h_t$\n({h_dim},)", ha='center', va='center', fontsize=12)

    # Route final h_T to Output Projection
    ax.arrow(8.5, 4.0, 0, -1.5, head_width=0.1, fc='k', ec='k', lw=1.5, linestyle='--')
    ax.text(8.6, 3.2, "At final step $T$", ha='left', va='center', fontsize=10, style='italic', color='gray')

    # Output Projection Layer (W_ho and b_o)
    ax.add_patch(patches.Rectangle((7.5, 1.5), 2.0, 1.0, color='salmon', ec='black', lw=2))
    ax.text(8.5, 2.0, f"Linear Proj.\n$W_{{ho}}$ ({h_dim}x{out_dim})\n$+ b_o$ ({out_dim},)", ha='center', va='center', fontsize=10)

    # Final Output
    ax.arrow(8.5, 1.5, 0, -0.8, head_width=0.1, fc='k', ec='k', lw=1.5)
    ax.text(8.5, 0.3, f"$\hat{{y}}$\n({out_dim},)", ha='center', va='center', fontsize=12)

    plt.title("Pure JAX Vanilla RNN Architecture & Matrix Dimensions", fontsize=14, y=1.05)
    plt.xlim(0, 10)
    plt.ylim(0, 6)
    plt.tight_layout()
    plt.show()

# Render the architecture using our initialized parameter dictionary
render_pure_rnn(rnn_params)

##Train, Execute, and Evaluate Basic RNN

Now we will train this manual JAX implementation. Because our parameters are stored in a standard Python dictionary, we use jax.value_and_grad to automatically compute the derivatives of the loss function with respect to every matrix and bias in `rnn_params`.

We will use `optax` to apply the Adam optimization algorithm.

In [ ]:
# --- Training Configuration ---
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 0.005

# Initialize Optax optimizer
optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(rnn_params)

# --- Define the Training Step ---
@jax.jit
def rnn_train_step(params, opt_state, x_batch, y_batch):
    def loss_fn(p):
        # jax.vmap applies our single-sequence forward pass across the entire batch
        preds = jax.vmap(rnn_forward, in_axes=(None, 0))(p, x_batch)
        # Mean Squared Error
        return jnp.mean(optax.l2_loss(preds, y_batch))

    # Calculate loss and gradients
    loss, grads = jax.value_and_grad(loss_fn)(params)

    # Calculate weight updates using Optax
    updates, opt_state = optimizer.update(grads, opt_state, params)

    # Apply the updates to the parameters
    new_params = optax.apply_updates(params, updates)

    return new_params, opt_state, loss

# --- Training Loop ---
print("Training Pure JAX Vanilla RNN...")
n_batches = len(X_train) // BATCH_SIZE
rnn_history = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i in range(n_batches):
        batch_x = X_train[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        batch_y = Y_train[i*BATCH_SIZE : (i+1)*BATCH_SIZE]

        rnn_params, opt_state, loss = rnn_train_step(rnn_params, opt_state, batch_x, batch_y)
        epoch_loss += loss.item()

    avg_loss = epoch_loss / n_batches
    rnn_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

# --- Evaluation ---
@jax.jit
def rnn_evaluate(params, X, Y):
    preds = jax.vmap(rnn_forward, in_axes=(None, 0))(params, X)
    mse = jnp.mean(jnp.square(preds - Y))
    return mse, preds

rnn_mse, rnn_preds = rnn_evaluate(rnn_params, X_test, Y_test)
print(f"Vanilla RNN Test MSE: {rnn_mse:.4f}")

##Basic Recurrent Neural Network (RNN) in Equinox


**Mathematical Formulation**

A vanilla RNN processes a sequence step-by-step, maintaining a hidden state $h_t$.Given input $x_t \in \mathbb{R}^d$ and previous state $h_{t-1} \in \mathbb{R}^h$, the update rule is:$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$The final hidden state $h_T$ summarizes the sequence and is passed through a linear layer to generate the predictions $\hat{y}$.

In Equinox, models are defined as `eqx.Module` classes. Attributes are automatically registered as parameters if they are JAX arrays.

In [ ]:
# --- 1. Define the Vanilla RNN ---

class VanillaRNN(eqx.Module):
    # Removed the unused `cell` attribute to satisfy Equinox initialization
    linear_ih: eqx.nn.Linear
    linear_hh: eqx.nn.Linear
    output_layer: eqx.nn.Linear
    hidden_size: int

    def __init__(self, input_size, hidden_size, output_size, key):
        keys = jax.random.split(key, 3)
        self.hidden_size = hidden_size
        self.linear_ih = eqx.nn.Linear(input_size, hidden_size, key=keys[0])
        self.linear_hh = eqx.nn.Linear(hidden_size, hidden_size, use_bias=False, key=keys[1])
        self.output_layer = eqx.nn.Linear(hidden_size, output_size, key=keys[2])

    def __call__(self, x):
        # x shape: (Sequence_Length, Features)

        # Initialize hidden state
        h = jnp.zeros((self.hidden_size,))

        # Unroll the sequence
        def scan_fn(h_prev, x_t):
            h_next = jax.nn.tanh(self.linear_ih(x_t) + self.linear_hh(h_prev))
            return h_next, h_next

        # jax.lax.scan is highly optimized for loops in JAX
        h_final, _ = jax.lax.scan(scan_fn, h, x)

        # Project final hidden state to output predictions
        return self.output_layer(h_final)

# --- 2. Render the Architecture ---
def render_rnn_arch():
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axis('off')

    # Draw Unrolled RNN
    for t in range(3):
        x_pos = t * 2
        # Input
        ax.add_patch(plt.Rectangle((x_pos, 0), 1, 0.5, color='lightblue'))
        ax.text(x_pos+0.5, 0.25, f'$x_{t+1}$', ha='center', va='center')
        # Hidden
        ax.add_patch(plt.Rectangle((x_pos, 1.5), 1, 0.8, color='lightgreen'))
        ax.text(x_pos+0.5, 1.9, f'$h_{t+1}$', ha='center', va='center')

        # Arrows (Input to Hidden)
        ax.arrow(x_pos+0.5, 0.5, 0, 0.8, head_width=0.1, color='black')

        # Arrows (Hidden to Hidden)
        if t < 2:
            ax.arrow(x_pos+1, 1.9, 0.8, 0, head_width=0.1, color='red')

    # Output from last hidden
    ax.arrow(4.5, 2.3, 0, 0.5, head_width=0.1, color='black')
    ax.add_patch(plt.Rectangle((4, 2.8), 1, 0.5, color='salmon'))
    ax.text(4.5, 3.05, f'$\hat{{y}}$', ha='center', va='center')

    plt.title("Vanilla RNN (Many-to-Vector) Architecture", fontsize=14)
    plt.show()

render_rnn_arch()

##Train, Execute, and Evaluate Basic RNN

Equinox interacts seamlessly with Optax. We use `eqx.filter_value_and_grad` to calculate gradients. This function is incredibly powerful: it automatically separates the differentiable arrays (weights) from non-differentiable elements (like strings or booleans) in your PyTree.

In [ ]:
# --- Training Configuration ---
BATCH_SIZE = 128
EPOCHS = 5
LEARNING_RATE = 0.005

key = jax.random.PRNGKey(42)
rnn_model = VanillaRNN(input_size=4, hidden_size=32, output_size=OUTPUT_STEPS, key=key)
optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(eqx.filter(rnn_model, eqx.is_array))

# --- Loss and Step Functions ---
@eqx.filter_value_and_grad
def compute_loss(model, x_batch, y_batch):
    # Vectorize the model over the batch dimension
    preds = jax.vmap(model)(x_batch)
    return jnp.mean(optax.l2_loss(preds, y_batch))

@eqx.filter_jit
def make_step(model, opt_state, x_batch, y_batch):
    loss, grads = compute_loss(model, x_batch, y_batch)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

# --- Training Loop ---
def train_model(model, opt_state, X, Y, epochs):
    n_batches = len(X) // BATCH_SIZE
    history = []

    for epoch in range(epochs):
        epoch_loss = 0.0
        for i in range(n_batches):
            batch_x = X[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            batch_y = Y[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
            model, opt_state, loss = make_step(model, opt_state, batch_x, batch_y)
            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_batches
        history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

    return model, history

print("Training Vanilla RNN...")
rnn_model, rnn_history = train_model(rnn_model, opt_state, X_train, Y_train, EPOCHS)

# --- Evaluation ---
@eqx.filter_jit
def evaluate(model, X, Y):
    preds = jax.vmap(model)(X)
    mse = jnp.mean(jnp.square(preds - Y))
    return mse, preds

rnn_mse, rnn_preds = evaluate(rnn_model, X_test, Y_test)
print(f"Vanilla RNN Test MSE: {rnn_mse:.4f}")

#Part C: LSTM Encoder-Decoder Architecture

The Vanilla RNN struggles to remember information from the start of the 24-step sequence due to vanishing gradients. Long Short-Term Memory (LSTM) cells introduce a cell state $C_t$ and gating mechanisms (Forget $f_t$, Input $i_t$, Output $o_t$) to protect and control information flow:

$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$$$C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$$

Instead of a simple linear layer at the end, we will build an Encoder-Decoder:

1. **Encoder:** Processes the 24 historical steps into a context vector (final $h$ and $C$).

2. **Decoder:** Uses the context vector to generate the 6 future steps recursively.

##Pure JAX LSTM Encoder-Decoder (From Scratch)

Before using a high-level library, we will implement the Long Short-Term Memory (LSTM) cell mathematically from scratch.

**The Mathematics of the LSTM Cell**

Unlike a vanilla RNN, an LSTM maintains two states: the **Hidden State** ($h_t$) and the **Cell State** ($C_t$). The Cell State acts like a conveyor belt, allowing gradients to flow unhindered across long sequences, solving the vanishing gradient problem.

Information is added or removed from the Cell State via three distinct gates (Forget, Input, Output), which are composed of sigmoid ($\sigma$) neural networks.

Given input $x_t$ and previous hidden state $h_{t-1}$, we concatenate them or compute their linear transformations simultaneously. For computational efficiency, we typically compute all four core transformations (the 3 gates + the candidate cell state) in a single matrix multiplication:

$$\text{Gates} = W \cdot x_t + U \cdot h_{t-1} + b$$

We split this resulting vector into four equal parts: $i, f, g, o$.

1. **Forget Gate** ($f_t$): Decides what to discard from the past cell state.$$f_t = \sigma(f)$$

2. **Input Gate** ($i_t$): Decides which new information to update.$$i_t = \sigma(i)$$

3. **Candidate Cell State** ($\tilde{C}_t$): Creates a vector of new candidate values.$$\tilde{C}_t = \tanh(g)$$

4. **Cell State Update** ($C_t$): The new cell state.$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

5. **Output Gate** ($o_t$): Decides what part of the cell state makes it to the hidden state.$$o_t = \sigma(o)$$

6. **Hidden State Update** ($h_t$):$$h_t = o_t \odot \tanh(C_t)$$

In [ ]:
# --- 1. Pure JAX LSTM Initialization ---
def init_lstm_params(input_size, hidden_size, key):
    k1, k2 = jax.random.split(key)
    # We stack the weights for the 4 internal transformations (i, f, g, o)
    # This allows 1 matrix multiplication instead of 4
    W = jax.random.normal(k1, (input_size, 4 * hidden_size)) * jnp.sqrt(2.0 / input_size)
    U = jax.random.normal(k2, (hidden_size, 4 * hidden_size)) * jnp.sqrt(2.0 / hidden_size)
    b = jnp.zeros(4 * hidden_size)

    # Standard practice: initialize forget gate bias to 1.0 to encourage remembering early in training
    b = b.at[hidden_size : 2 * hidden_size].set(1.0)

    return {'W': W, 'U': U, 'b': b}

def init_seq2seq_params(input_size, hidden_size, key):
    k1, k2, k3 = jax.random.split(key, 3)
    return {
        'encoder': init_lstm_params(input_size, hidden_size, k1),
        'decoder': init_lstm_params(1, hidden_size, k2), # Decoder takes only past temperature
        'proj_w': jax.random.normal(k3, (hidden_size, 1)) * jnp.sqrt(2.0 / hidden_size),
        'proj_b': jnp.zeros(1)
    }

# --- 2. Pure JAX LSTM Cell Logic ---
def lstm_cell(params, state, x):
    h, c = state
    W, U, b = params['W'], params['U'], params['b']

    # Single unified linear transformation
    gates = jnp.dot(x, W) + jnp.dot(h, U) + b

    # Split into the 4 components
    i, f, g, o = jnp.split(gates, 4)

    # Apply non-linearities
    i = jax.nn.sigmoid(i)
    f = jax.nn.sigmoid(f)
    g = jax.nn.tanh(g)
    o = jax.nn.sigmoid(o)

    # Update Cell and Hidden States
    c_next = f * c + i * g
    h_next = o * jax.nn.tanh(c_next)

    return (h_next, c_next), h_next

# --- 3. Seq2Seq Forward Pass ---
def pure_seq2seq_forward(params, x, output_steps):
    hidden_size = params['encoder']['U'].shape[0]

    # 1. ENCODER
    def encode_step(state, x_t):
        return lstm_cell(params['encoder'], state, x_t)

    init_enc_state = (jnp.zeros(hidden_size), jnp.zeros(hidden_size))
    # scan over the sequence dimension (0th axis by default)
    final_enc_state, _ = jax.lax.scan(encode_step, init_enc_state, x)

    # 2. DECODER
    def decode_step(state, _):
        h, c, dec_in = state
        # Process step
        (h_next, c_next), _ = lstm_cell(params['decoder'], (h, c), dec_in)
        # Project hidden state to output prediction
        pred = jnp.dot(h_next, params['proj_w']) + params['proj_b']
        return (h_next, c_next, pred), pred

    # Start decoding with the last input temperature
    dec_input_0 = x[-1, 1:2]
    init_dec_state = (final_enc_state[0], final_enc_state[1], dec_input_0)

    _, predictions = jax.lax.scan(decode_step, init_dec_state, None, length=output_steps)
    return predictions.squeeze()

# --- 4. Training Loop for Pure JAX Model ---
key = jax.random.PRNGKey(101)
pure_params = init_seq2seq_params(input_size=4, hidden_size=32, key=key)
pure_optimizer = optax.adam(0.005)
pure_opt_state = pure_optimizer.init(pure_params)

@jax.jit
def pure_train_step(params, opt_state, x_batch, y_batch):
    def loss_fn(p):
        # vmap over the batch dimension
        preds = jax.vmap(pure_seq2seq_forward, in_axes=(None, 0, None))(p, x_batch, OUTPUT_STEPS)
        return jnp.mean(optax.l2_loss(preds, y_batch))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = pure_optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state, loss

print("Training Pure JAX LSTM Encoder-Decoder...")
n_batches = len(X_train) // BATCH_SIZE
pure_history = []

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for i in range(n_batches):
        batch_x = X_train[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        batch_y = Y_train[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
        pure_params, pure_opt_state, loss = pure_train_step(pure_params, pure_opt_state, batch_x, batch_y)
        epoch_loss += loss.item()

    avg_loss = epoch_loss / n_batches
    pure_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

###Rendering the Internal LSTM Architecture

To truly understand the pure JAX implementation above, let's render the internal wiring of an LSTM cell, detailing how the inputs, previous states, and the 4 gates interact within the Encoder-Decoder pipeline.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def render_dynamic_seq2seq_detailed(params):
    """
    Renders the Seq2Seq architecture dynamically, showing the full internal
    LSTM gating mechanisms and state routing based on the JAX parameter dictionary.
    """
    # 1. Extract dimensions directly from the weight matrices
    enc_in_size = params['encoder']['W'].shape[0]
    hidden_size = params['encoder']['W'].shape[1] // 4
    dec_in_size = params['decoder']['W'].shape[0]
    out_size = params['proj_w'].shape[1]

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.axis('off')

    def draw_lstm_block(x, y, w, h, title, in_dim, color):
        """Helper to draw a detailed LSTM cell at a specific location."""
        # Main Cell Box
        ax.add_patch(patches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                                            ec="black", fc=color, lw=2, alpha=0.8, zorder=1))

        # Title & Dimensions
        ax.text(x + w/2, y + h + 0.25, f"{title}\nInput: {in_dim}d  |  Hidden: {hidden_size}d",
                ha='center', va='bottom', fontsize=12, weight='bold')

        # --- State Conveyor Belts ---
        c_y = y + h * 0.8  # Cell State (Top)
        h_y = y + h * 0.2  # Hidden State (Bottom)

        # Draw the continuous lines for states
        ax.plot([x - 0.5, x + w + 0.5], [c_y, c_y], 'k-', lw=3, zorder=2)
        ax.plot([x - 0.5, x + w + 0.5], [h_y, h_y], 'k-', lw=3, zorder=2)

        # --- Internal Gates ---
        gate_names = ['Forget\n$f_t$', 'Input\n$i_t$', 'Cand.\n$\tilde{C}_t$', 'Output\n$o_t$']
        ops = ['$\sigma$', '$\sigma$', 'tanh', '$\sigma$']
        spacing = w / 5
        gate_y = y + h * 0.45

        for i, (name, op) in enumerate(zip(gate_names, ops)):
            gx = x + (i + 1) * spacing
            # Gate Node
            ax.add_patch(patches.Circle((gx, gate_y), 0.2, fc='yellow', ec='black', zorder=3))
            ax.text(gx, gate_y, op, ha='center', va='center', zorder=4, fontsize=11, weight='bold')
            ax.text(gx, gate_y - 0.4, name, ha='center', va='top', fontsize=9)

            # Connections from bottom (combining h_{t-1} and x_t)
            ax.plot([gx, gx], [h_y, gate_y - 0.2], 'k--', lw=1.5, zorder=2)

        # --- Operations on the Cell State (C_t) ---

        # 1. Forget Multiply
        fx = x + spacing
        ax.add_patch(patches.Circle((fx, c_y), 0.15, fc='pink', ec='black', zorder=3))
        ax.text(fx, c_y, 'X', ha='center', va='center', zorder=4, fontsize=9, weight='bold')
        ax.arrow(fx, gate_y + 0.2, 0, (c_y - gate_y) - 0.35, head_width=0.08, fc='k', ec='k', zorder=2)

        # 2. Input & Candidate Multiply
        ix = x + 2 * spacing
        cx = x + 3 * spacing
        mx = (ix + cx) / 2
        ax.add_patch(patches.Circle((mx, y + h * 0.62), 0.15, fc='pink', ec='black', zorder=3))
        ax.text(mx, y + h * 0.62, 'X', ha='center', va='center', zorder=4, fontsize=9, weight='bold')
        ax.plot([ix, mx], [gate_y + 0.2, y + h * 0.62], 'k-', lw=1.5, zorder=2)
        ax.plot([cx, mx], [gate_y + 0.2, y + h * 0.62], 'k-', lw=1.5, zorder=2)

        # 3. Add to Cell State
        ax.add_patch(patches.Circle((mx, c_y), 0.15, fc='lightgreen', ec='black', zorder=3))
        ax.text(mx, c_y, '+', ha='center', va='center', zorder=4, fontsize=12, weight='bold')
        ax.arrow(mx, y + h * 0.62 + 0.15, 0, (c_y - (y + h * 0.62)) - 0.3, head_width=0.08, fc='k', ec='k', zorder=2)

        # --- Operations on the Hidden State (h_t) ---

        # 4. Tanh of Cell State & Multiply with Output Gate
        ox = x + 4 * spacing

        # Tanh node on the C_t line
        ax.add_patch(patches.Circle((ox, c_y), 0.2, fc='white', ec='black', zorder=3))
        ax.text(ox, c_y, 'tanh', ha='center', va='center', zorder=4, fontsize=9)

        # Multiply node on the h_t line
        ax.add_patch(patches.Circle((ox, h_y), 0.15, fc='pink', ec='black', zorder=3))
        ax.text(ox, h_y, 'X', ha='center', va='center', zorder=4, fontsize=9, weight='bold')

        # Route C_t down to the multiply node
        ax.arrow(ox, c_y - 0.2, 0, -(c_y - h_y - 0.35), head_width=0.08, fc='k', ec='k', zorder=2)
        # Route Output gate to the multiply node
        ax.arrow(ox, gate_y - 0.2, 0, -(gate_y - h_y - 0.35), head_width=0.08, fc='k', ec='k', zorder=2)

    # ==========================================
    # 2. Build the Full Seq2Seq Graphic
    # ==========================================

    # A. Encoder Box
    draw_lstm_block(x=2, y=2, w=4, h=3, title="Encoder", in_dim=enc_in_size, color="#e6f2ff")

    # Encoder Input
    ax.arrow(4, 0.5, 0, 1.5, head_width=0.15, fc='k', ec='k', lw=2)
    ax.text(4, 0.2, f"Input Sequence $X_t$\nshape: ({enc_in_size},)", ha='center', va='center', fontsize=12)

    # Initial States (Encoder)
    ax.text(1.2, 4.4, "$C_{0}=\mathbf{0}$", ha='right', va='center', fontsize=11)
    ax.text(1.2, 2.6, "$h_{0}=\mathbf{0}$", ha='right', va='center', fontsize=11)

    # B. Context Vectors (Bridge from Encoder to Decoder)
    ax.arrow(6.5, 4.4, 2.0, 0, head_width=0.15, fc='k', ec='k', lw=2)
    ax.text(7.5, 4.55, "Context $C_{enc}$", ha='center', va='bottom', weight='bold', color='blue')

    ax.arrow(6.5, 2.6, 2.0, 0, head_width=0.15, fc='k', ec='k', lw=2)
    ax.text(7.5, 2.75, "Context $h_{enc}$", ha='center', va='bottom', weight='bold', color='blue')

    # C. Decoder Box
    draw_lstm_block(x=9, y=2, w=4, h=3, title="Decoder", in_dim=dec_in_size, color="#e6ffe6")

    # Decoder Input (Feedback)
    ax.arrow(11, 0.5, 0, 1.5, head_width=0.15, fc='k', ec='k', lw=2)
    ax.text(11, 0.2, f"Feedback $\hat{{y}}_{{t-1}}$\nshape: ({dec_in_size},)", ha='center', va='center', fontsize=12)

    # D. Output & Projection
    ax.text(13.5, 4.4, "$C_{dec}$ (Discarded)", ha='left', va='center', fontsize=10, style='italic', color='gray')

    # Route final Decoder hidden state to the Linear layer
    ax.arrow(13.5, 2.6, 1.0, 0, head_width=0.15, fc='k', ec='k', lw=2)

    # Linear Projection Box
    ax.add_patch(patches.Rectangle((14.5, 2.1), 1.5, 1.0, color='salmon', ec='black', lw=2))
    ax.text(15.25, 2.6, f"Linear\nProj.\n({hidden_size} $\\rightarrow$ {out_size})", ha='center', va='center', weight='bold')

    # Final Prediction Arrow
    ax.arrow(16.0, 2.6, 0.8, 0, head_width=0.1, fc='k', ec='k', lw=2)
    ax.text(17.2, 2.6, f"$\hat{{y}}_t$", ha='center', va='center', fontsize=16, weight='bold')

    plt.title("Detailed Internal LSTM Wiring derived from JAX Parameters", fontsize=16, weight='bold')
    plt.xlim(0, 18)
    plt.ylim(0, 6.5)
    plt.tight_layout()
    plt.show()

# Execute the detailed renderer
render_dynamic_seq2seq_detailed(pure_params)

##Transitioning to Equinox - Neural Networks as PyTrees


Writing neural networks from scratch in pure JAX (as we did in Part E) provides unparalleled transparency into the linear algebra and state routing. However, as models scale to millions of parameters and include complex sub-modules like Attention heads or deep residual blocks, manually managing nested parameter dictionaries (params`['encoder']['U']`) and writing explicit initialization functions becomes tedious and error-prone.


Enter **Equinox**.

Created to bridge the gap between PyTorch's intuitive, object-oriented design and JAX's strict functional requirements, Equinox allows you to define neural networks using standard Python classes without breaking JAX's JIT compilation or Autograd.


**The Core Concept: Callable PyTrees**

In JAX, functions must be pure, and all state (like weights and biases) must be explicitly passed in and out. JAX manages this via PyTrees—nested structures of lists, tuples, and dictionaries.

Equinox's brilliant innovation is its base class: `eqx.Module`. When you subclass `eqx.Module`, Equinox automatically registers your Python class as a JAX PyTree.

* The leaves of the tree are the class attributes (your weights, biases, and sub-modules).

* The class itself remains strictly functional under the hood, meaning you can safely pass the entire model object directly into `@jax.jit` or `jax.grad`.

**Mapping our LSTM Architecture to Equinox**

In the next code cell, we will rebuild our Sequence-to-Sequence model using Equinox. Notice how the boilerplate disappears, yet the architectural flow remains the same:

1. **Modules instead of Matrices:** Instead of defining $W$, $U$, and $b$ manually, we instantiate two `eqx.nn.LSTMCell` objects—one for the Encoder and one for the Decoder—and an `eqx.nn.Linear` layer for the output projection.

2. **Automatic Initialization:** We simply pass a `jax.random.PRNGKey` to the module constructors, and Equinox handles the shape math and weight initializations internally.

3. **The Forward Pass** (`__call__`): Just like in our pure JAX implementation, Equinox's LSTMCell processes exactly one time step at a time. We will still use `jax.lax.scan` to unroll the cell over our 24-step input sequence, explicitly passing the $(h_t, C_t)$ tuple from step to step.

Let's see how clean the code becomes when the framework handles the parameter tracking for us.

In [ ]:
# --- 1. Define the LSTM Encoder-Decoder ---
class LSTMSeq2Seq(eqx.Module):
    encoder: eqx.nn.LSTMCell
    decoder: eqx.nn.LSTMCell
    output_layer: eqx.nn.Linear
    hidden_size: int
    output_steps: int

    def __init__(self, input_size, hidden_size, output_size, output_steps, key):
        keys = jax.random.split(key, 3)
        self.hidden_size = hidden_size
        self.output_steps = output_steps

        # Encoder takes the 4 input features
        self.encoder = eqx.nn.LSTMCell(input_size, hidden_size, key=keys[0])

        # Decoder takes its previous prediction (1 feature: temperature) as input
        self.decoder = eqx.nn.LSTMCell(1, hidden_size, key=keys[1])

        # Output layer maps hidden state to temperature prediction
        self.output_layer = eqx.nn.Linear(hidden_size, 1, key=keys[2])

    def __call__(self, x):
        # x shape: (Sequence_Length, Features)

        # 1. ENCODER
        h = jnp.zeros((self.hidden_size,))
        c = jnp.zeros((self.hidden_size,))

        def encode_step(state, x_t):
            h, c = state
            h_new, c_new = self.encoder(x_t, (h, c))
            return (h_new, c_new), None

        (h, c), _ = jax.lax.scan(encode_step, (h, c), x)

        # 2. DECODER
        # Start decoding with the last known temperature (index 1 of the last input step)
        dec_input = x[-1, 1:2]

        def decode_step(state, _):
            h, c, dec_in = state
            h_new, c_new = self.decoder(dec_in, (h, c))
            pred = self.output_layer(h_new)
            return (h_new, c_new, pred), pred

        # We scan for `output_steps` times. The array passed is just for length mapping.
        _, predictions = jax.lax.scan(decode_step, (h, c, dec_input), jnp.arange(self.output_steps))

        # Shape output to match targets (Output_Steps,)
        return predictions.squeeze()

# --- 2. Render Architecture ---
def render_seq2seq_arch():
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis('off')

    # Encoder Box
    ax.add_patch(plt.Rectangle((1, 1), 3, 2, color='plum'))
    ax.text(2.5, 2, "LSTM Encoder\n(Reads $x_1...x_{24}$)", ha='center', va='center', fontsize=12)

    # Context Vector Arrow
    ax.arrow(4, 2, 1.5, 0, head_width=0.15, color='black', linewidth=2)
    ax.text(4.75, 2.2, "Context\n$(h_{24}, C_{24})$", ha='center', va='center')

    # Decoder Box
    ax.add_patch(plt.Rectangle((5.5, 1), 3, 2, color='khaki'))
    ax.text(7.0, 2, "LSTM Decoder\n(Generates $\hat{y}_1...\hat{y}_6$)", ha='center', va='center', fontsize=12)

    plt.title("LSTM Seq2Seq Architecture", fontsize=14)
    plt.show()

render_seq2seq_arch()

##Train, Execute, and Evaluate the Equinox LSTM

Now we train the LSTM-based Sequence-to-Sequence model using the exact same Optax setup and compare its performance to the Vanilla RNN.

In [ ]:
# --- Setup LSTM Model ---
lstm_model = LSTMSeq2Seq(input_size=4, hidden_size=32, output_size=1, output_steps=OUTPUT_STEPS, key=key)
opt_state_lstm = optimizer.init(eqx.filter(lstm_model, eqx.is_array))

# Re-using the same training functions, just passing the new model
print("Training LSTM Seq2Seq...")
lstm_model, lstm_history = train_model(lstm_model, opt_state_lstm, X_train, Y_train, EPOCHS)

# --- Evaluation ---
lstm_mse, lstm_preds = evaluate(lstm_model, X_test, Y_test)
print(f"LSTM Seq2Seq Test MSE: {lstm_mse:.4f}")

# --- Plot Results ---
plt.figure(figsize=(12, 5))

# Loss curve comparison
plt.subplot(1, 2, 1)
plt.plot(rnn_history, label='Vanilla RNN')
plt.plot(lstm_history, label='LSTM Seq2Seq')
plt.title('Training Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Visualizing a Prediction
plt.subplot(1, 2, 2)
sample_idx = 200 # Look at one random test sample

# The past 24 hours of temperature (target index 1)
past_history = X_test[sample_idx, :, TARGET_IDX]
true_future = Y_test[sample_idx, :]
rnn_prediction = rnn_preds[sample_idx, :]
lstm_prediction = lstm_preds[sample_idx, :]

# Create time axes
t_past = np.arange(-INPUT_STEPS, 0)
t_future = np.arange(0, OUTPUT_STEPS)

plt.plot(t_past, past_history, label='History', marker='.')
plt.plot(t_future, true_future, label='True Future', marker='o', color='green')
plt.plot(t_future, rnn_prediction, label='RNN Pred', marker='x', color='red', linestyle='--')
plt.plot(t_future, lstm_prediction, label='LSTM Pred', marker='^', color='purple', linestyle='--')

plt.title('Prediction Comparison on Sample Window')
plt.xlabel('Time Steps')
plt.ylabel('Standardized Temperature')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. Evaluate the Pure JAX Model ---
@jax.jit
def pure_evaluate(params, X, Y):
    # Vectorize the forward pass over the batch dimension
    preds = jax.vmap(pure_seq2seq_forward, in_axes=(None, 0, None))(params, X, OUTPUT_STEPS)
    mse = jnp.mean(jnp.square(preds - Y))
    return mse, preds

print("Evaluating Pure JAX Model...")
pure_mse, pure_preds = pure_evaluate(pure_params, X_test, Y_test)

print("-" * 30)
print(f"Equinox LSTM Test MSE:   {lstm_mse:.4f}") # from Part G
print(f"Pure JAX LSTM Test MSE:  {pure_mse:.4f}")
print("-" * 30)

# --- 2. Visual Comparison: Pure JAX vs. Equinox ---
plt.figure(figsize=(10, 5))

sample_idx = 200 # Re-using the same sample from earlier

# Time axes
t_past = np.arange(-INPUT_STEPS, 0)
t_future = np.arange(0, OUTPUT_STEPS)

# Extract sequences
past_history = X_test[sample_idx, :, TARGET_IDX]
true_future = Y_test[sample_idx, :]
equinox_prediction = lstm_preds[sample_idx, :]
pure_prediction = pure_preds[sample_idx, :]

# Plotting
plt.plot(t_past, past_history, label='History', marker='.', color='gray')
plt.plot(t_future, true_future, label='True Future', marker='o', color='green', linewidth=2)
plt.plot(t_future, equinox_prediction, label='Equinox LSTM', marker='^', color='purple', linestyle='--', alpha=0.8)
plt.plot(t_future, pure_prediction, label='Pure JAX LSTM', marker='x', color='orange', linestyle='--', alpha=0.8)

plt.axvline(0, color='black', linestyle='-', linewidth=1)
plt.title('Prediction Overlay: High-Level API vs. From-Scratch Implementation')
plt.xlabel('Time Steps')
plt.ylabel('Standardized Temperature')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#Final Thoughts

By isolating the cell logic and architectural flow in JAX and Equinox, you can see how the LSTM's state passing ($h$ and $C$) provides a stronger foundation for the sequence-to-sequence generation compared to a single projection from a Vanilla RNN.